# 01b — Diagnose the PushT reproduction gap

**Why.** Notebook 01 gave PushT 89.3 % (150 episodes) against the paper's 96.0 ± 2.83,
outside the ±5 band. This notebook runs two cheap diagnostics; neither trains anything.

- **A. Dataset-replay ceiling.** Same episodes, same start/goal states and success test as
  the official evaluation, but instead of planning we replay the dataset's own actions for
  25 steps. If the simulator + dataset were perfectly consistent this would succeed ~100 %.
  Episodes where replay fails cannot be blamed on the world model. TwoRoom is run as a control.
- **B. More planner episodes.** The official evaluation on extra seeds, to tighten the
  estimate of what the released PushT checkpoint achieves in this software environment.
  These seeds are *not* protocol seeds; results go to `results/diagnostics/`, never into
  the reproduction table.

**Expected runtime.** ~20 min (≈8 min downloads; each evaluation run ≈1 min, mostly start-up).
Extrapolated from notebook 01; replay runs are unmeasured.

**Expected compute.** ≤ 0.4 L4-hours.

## 1. Config (the only cell you edit)

In [ ]:
REPO_URL = "https://github.com/CHANGE_ME/jepa-memory.git"  # <-- your remote
REPO_REF = "main"
REPO_DIR = "/content/jepa-memory"

REPLAY_ENVS = ["pusht", "tworoom"]      # A: replay ceiling (tworoom = control)
PLAN_ENV = "pusht"                      # B: more planner episodes
EXTRA_SEEDS = [45, 46, 47, 48, 49, 50, 51]   # B: 7 x 50 = 350 further episodes
INSTALL_EXTRA = "wm,dev"
FORCE = False

RUN_ID = "nb01b-diagnose-pusht"
NOTEBOOK = "01b_diagnose_pusht"
EST_HOURS = 0.4
SEED = 0
EXPECTED_GPUS = ("L4", "A100")

## 2. GPU check

In [ ]:
import subprocess

try:
    gpu_name = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    gpu_name = ""
print("GPU:", gpu_name or "NONE")
HAS_GPU = bool(gpu_name)
if not any(g in gpu_name for g in EXPECTED_GPUS):
    print(f"WARNING: expected one of {EXPECTED_GPUS}. The official eval script needs CUDA "
          "(le-wm/eval.py:89); budget accounting assumes L4 or A100.")

## 3. Drive mount

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not on Colab: using repo-relative paths.")

## 4. Clone / update the repo (with the pinned `le-wm` submodule)

In [ ]:
def _git(*args, cwd=None):
    subprocess.run(["git", *args], cwd=cwd, check=True)

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    assert "CHANGE_ME" not in REPO_URL, "Set REPO_URL in the config cell."
    clone_url = REPO_URL
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")
            if token:
                clone_url = REPO_URL.replace("https://", f"https://{token}@")
        except Exception:
            pass  # no secret configured -> public clone
    subprocess.run(["git", "clone", "-q", clone_url, REPO_DIR], check=True, capture_output=True)
    _git("remote", "set-url", "origin", REPO_URL, cwd=REPO_DIR)  # never keep the token on disk
else:
    _git("fetch", "-q", "--all", cwd=REPO_DIR)
_git("checkout", "-q", REPO_REF, cwd=REPO_DIR)
on_branch = subprocess.run(["git", "symbolic-ref", "-q", "HEAD"], cwd=REPO_DIR,
                           capture_output=True).returncode == 0
if on_branch:
    subprocess.run(["git", "pull", "-q", "--ff-only"], cwd=REPO_DIR)  # ok to fail offline
_git("submodule", "update", "--init", "--recursive", cwd=REPO_DIR)
os.chdir(REPO_DIR)
print("repo at", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["git", "submodule", "status"], capture_output=True, text=True).stdout.strip())

## 5. Install pinned dependencies and the `jmem` package

In [ ]:
# torch is NOT reinstalled: Colab's CUDA build satisfies torch>=2.4,<3 (see pyproject.toml).
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f".[{INSTALL_EXTRA}]"])
assert r.returncode == 0, "pip install failed (scroll up for the error)"
src = os.path.join(REPO_DIR, "src")
if src not in sys.path:
    sys.path.insert(0, src)  # editable installs are only picked up by new processes
import jmem
print("jmem", jmem.__version__)

## 6. Paths, seeds, ledger

In [ ]:
import json, shutil
from jmem import paths, ledger, utils, datasets, aggregate
from jmem.evaluate import run_official_eval, check_protocol, load_protocol, success_rate

assert HAS_GPU, "Part B needs a CUDA runtime (le-wm/eval.py:89)."
P = paths.setup()
utils.seed_everything(SEED)
ledger.start(RUN_ID, est_hours=EST_HOURS, notebook=NOTEBOOK)

protocol = load_protocol()
NUM_EVAL = protocol["common"]["num_eval"]
PROTOCOL_SEEDS = protocol["eval_seeds"]
assert not set(EXTRA_SEEDS) & set(PROTOCOL_SEEDS), "EXTRA_SEEDS must not repeat protocol seeds"
OFF = str(protocol["common"]["goal_offset_steps"])
OUT = P.results / "diagnostics"
REPO_OUT = paths.repo_root() / "results" / "diagnostics"
REPRO = paths.repo_root() / "results" / "reproduction"
RAW = OUT / "raw"
for d in (OUT, REPO_OUT, RAW):
    d.mkdir(parents=True, exist_ok=True)
for env in sorted(set(REPLAY_ENVS) | {PLAN_ENV}):
    for seed in PROTOCOL_SEEDS:
        assert (REPRO / f"official-{env}-released-s{seed}.json").exists(), "commit results/reproduction first"
utils.save_resolved_config({"replay_envs": REPLAY_ENVS, "plan_env": PLAN_ENV, "extra_seeds": EXTRA_SEEDS,
                            "num_eval": NUM_EVAL, "protocol": protocol}, OUT, name="nb01b")

## 7. Released checkpoints and datasets

In [ ]:
for env in sorted(set(REPLAY_ENVS) | {PLAN_ENV}):
    datasets.ensure_checkpoint(env)
    datasets.ensure_dataset(env, P.data_home)

## 8. Part A — dataset-replay ceiling (same episodes as the reproduction)

`mode="replay"` keeps `eval.py`'s episode sampling, state/goal setting and success test and
swaps the planner for an open-loop replay of the dataset actions over the 25 goal-offset steps.

In [ ]:
REPLAY, CROSS = {}, {}
for env in REPLAY_ENVS:
    plan_all, replay_all = [], []
    for seed in PROTOCOL_SEEDS:
        rec = run_official_eval(env, "unused", out_dir=RAW, num_eval=NUM_EVAL, seed=seed,
                                paths=P, force=FORCE, mode="replay")
        official = json.loads((REPRO / f"official-{env}-released-s{seed}.json").read_text())
        plan = official["episode_successes"][OFF]
        replay = [bool(x) for x in rec["metrics"]["episode_successes"]]
        assert len(plan) == len(replay) == NUM_EVAL
        assert rec["evaluate_kwargs"]["eval_budget"] == int(OFF) and rec["policy"]["type"] == "ReplayPolicy"
        REPLAY[(env, seed)] = {"replay": replay, "plan": plan,
                               "episodes_idx": rec["evaluate_kwargs"]["episodes_idx"],
                               "start_steps": rec["evaluate_kwargs"]["start_steps"]}
        plan_all += plan
        replay_all += replay
        print(f"{env} seed {seed}: replay {sum(replay)}/{len(replay)} | planner {sum(plan)}/{len(plan)}")
    CROSS[env] = aggregate.crosstab(plan_all, replay_all)
    print(env, "pooled:", CROSS[env])

## 9. Part B — official evaluation on extra seeds (PushT)

In [ ]:
EXTRA = {}
for seed in EXTRA_SEEDS:
    rec = run_official_eval(PLAN_ENV, datasets.CHECKPOINTS[PLAN_ENV], out_dir=RAW,
                            num_eval=NUM_EVAL, seed=seed, paths=P, force=FORCE)
    bad = check_protocol(rec, protocol, PLAN_ENV)
    # the only allowed deviation is the non-protocol seed itself
    assert all(m.startswith("seed:") for m in bad), bad
    eps = [bool(x) for x in rec["metrics"]["episode_successes"]]
    EXTRA[seed] = eps
    print(f"{PLAN_ENV} seed {seed}: planner {sum(eps)}/{len(eps)}")

## 10. Diagnosis summary (written to JSON; all numbers computed here)

In [ ]:
def pooled(bits):
    k, n = sum(bits), len(bits)
    lo, hi = aggregate.wilson_interval(k, n)
    return {"k": k, "n": n, "percent": 100 * k / n, "wilson95": [100 * lo, 100 * hi]}

proto_plan = [b for s in PROTOCOL_SEEDS for b in REPLAY[(PLAN_ENV, s)]["plan"]] if PLAN_ENV in REPLAY_ENVS else []
extra_plan = [b for s in EXTRA_SEEDS for b in EXTRA[s]]
per_seed = [100 * success_rate(EXTRA[s]) for s in EXTRA_SEEDS]
mean, half = aggregate.mean_ci_t(per_seed)
DIAG = {
    "replay": {env: {"pooled": pooled([b for s in PROTOCOL_SEEDS for b in REPLAY[(env, s)]["replay"]]),
                     "crosstab_vs_planner": CROSS[env],
                     "per_seed": {str(s): REPLAY[(env, s)] for s in PROTOCOL_SEEDS}}
               for env in REPLAY_ENVS},
    "planner_extra_seeds": {"env": PLAN_ENV, "seeds": EXTRA_SEEDS, "per_seed_percent": per_seed,
                            "mean_percent": mean, "ci_t95_half": half, "pooled": pooled(extra_plan),
                            "episode_successes": {str(s): EXTRA[s] for s in EXTRA_SEEDS}},
    "planner_all_seeds_pooled": pooled(proto_plan + extra_plan),
    "git_sha": utils.git_sha(), "versions": utils.env_versions(), "hardware": utils.detect_gpu(),
    "timestamp": utils.utc_timestamp(),
}
(OUT / "pusht_diagnosis.json").write_text(json.dumps(DIAG, indent=2))
shutil.copy2(OUT / "pusht_diagnosis.json", REPO_OUT / "pusht_diagnosis.json")
print(json.dumps({k: DIAG[k] for k in ("planner_all_seeds_pooled",)}, indent=1))

## 11. Validation

In [ ]:
import math

for env in REPLAY_ENVS:
    ct = CROSS[env]
    assert ct["n"] == NUM_EVAL * len(PROTOCOL_SEEDS)
    assert sum(v for k, v in ct.items() if k != "n") == ct["n"]
assert len(extra_plan) == NUM_EVAL * len(EXTRA_SEEDS)
for block in (DIAG["planner_all_seeds_pooled"], DIAG["planner_extra_seeds"]["pooled"]):
    assert 0.0 <= block["percent"] <= 100.0 and not math.isnan(block["percent"])
assert (REPO_OUT / "pusht_diagnosis.json").exists()
json.loads((REPO_OUT / "pusht_diagnosis.json").read_text())
print("all validation checks passed")

## 12. Ledger and summary (paste everything below back to Claude)

In [ ]:
row = ledger.stop()
budget = ledger.summary()
archive = shutil.make_archive(str(P.results / "diagnostics_results"), "zip", REPO_OUT)

print("\n===== NB01b SUMMARY (paste back) =====")
print("repo_sha   :", utils.git_sha())
print("gpu        :", gpu_name)
print("versions   :", json.dumps(DIAG["versions"]))
print("ledger_row :", json.dumps(row))
print("ledger_used_L4h / projected:", round(budget.used, 3), "/", round(budget.projected, 3))
print("results zip:", archive, "(unzip into the repo's results/diagnostics/ and commit)")
for env in REPLAY_ENVS:
    r = DIAG["replay"][env]
    print(f"A replay {env}: {r['pooled']['k']}/{r['pooled']['n']} = {r['pooled']['percent']:.1f}% "
          f"wilson95 [{r['pooled']['wilson95'][0]:.1f}, {r['pooled']['wilson95'][1]:.1f}] | crosstab {r['crosstab_vs_planner']}")
    for s in PROTOCOL_SEEDS:
        d = REPLAY[(env, s)]
        print(f"    seed {s} replay bits  ", "".join("1" if b else "0" for b in d["replay"]))
        print(f"    seed {s} planner bits ", "".join("1" if b else "0" for b in d["plan"]))
e = DIAG["planner_extra_seeds"]
print(f"B planner {PLAN_ENV} extra seeds {EXTRA_SEEDS}: per seed {[round(x) for x in e['per_seed_percent']]} "
      f"| mean {e['mean_percent']:.1f} ± {e['ci_t95_half']:.1f} (t, 95%) | pooled {e['pooled']['k']}/{e['pooled']['n']} "
      f"= {e['pooled']['percent']:.1f}% wilson95 [{e['pooled']['wilson95'][0]:.1f}, {e['pooled']['wilson95'][1]:.1f}]")
a = DIAG["planner_all_seeds_pooled"]
print(f"B planner {PLAN_ENV} all {len(PROTOCOL_SEEDS) + len(EXTRA_SEEDS)} seeds pooled: {a['k']}/{a['n']} = {a['percent']:.1f}% "
      f"wilson95 [{a['wilson95'][0]:.1f}, {a['wilson95'][1]:.1f}]")